# 01 COCO 이미지/라벨 전달용 dataset packaging

COCO bbox JSON 안에 들어있는 이미지 절대경로와 annotation 정보를 이용해서, 다른 사람에게 전달하기 쉬운 `images/`, `labels/`, `annotations/` 구조로 복사 정리합니다. 기본값은 `DRY_RUN=True`, `RUN_PACK=False`라서 실제 파일을 만들지 않습니다.


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").is_file() and (path / "src").is_dir():
            return path
    raise RuntimeError("repo root를 찾지 못했습니다. notebook을 repo 안에서 실행하거나 REPO_ROOT를 직접 지정하세요.")


REPO_ROOT = find_repo_root(Path.cwd())
src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("REPO_ROOT:", REPO_ROOT)

from labelstudio_bbox_tools.dataset_pack.coco_pack import pack_coco_dataset


## 설정

`COCO_INPUTS`는 두 가지 방식 중 하나를 쓰면 됩니다.

1. split 직접 지정: `{"train": Path(...), "val": Path(...)}`
2. list 자동판정: `[Path(...train...json), Path(...val...json)]`

전달용 dataset은 기본적으로 이미지를 실제 복사합니다. symlink/hardlink는 같은 서버 내부에서만 편하므로, 다른 사람에게 넘길 때는 `COPY_MODE="copy"`를 권장합니다.


In [ ]:
# ===== 사용자가 주로 바꾸는 설정 =====

# split을 직접 지정하는 방식입니다. 경로는 본인 환경에 맞게 바꾸세요.
COCO_INPUTS = {
    "train": Path("/path/to/train_coco.json"),
    "val": Path("/path/to/val_coco.json"),
}

# 파일명에 train/val/test가 들어있으면 아래처럼 list로 둬도 자동 판정됩니다.
# COCO_INPUTS = [Path("/path/to/my_train.json"), Path("/path/to/my_val.json")]

OUT_DIR = Path("/path/to/output_dataset_for_handoff")

# label format 선택입니다. YOLO 학습용 전달이면 yolo를 권장합니다. COCO만 저장하려면 none으로 바꾸세요.
LABEL_FORMAT = "yolo"  # "yolo" 또는 "none"
SAVE_REWRITTEN_COCO = True  # True이면 annotations/train_coco.json, val_coco.json도 함께 만듭니다.

# 전달 목적이면 copy가 안전합니다. symlink/hardlink는 같은 서버/디스크 안에서만 편합니다.
COPY_MODE = "copy"  # "copy", "symlink", "hardlink"

# annotation 없는 이미지도 포함하고 빈 txt를 만들지 여부입니다. negative image가 필요하면 True를 유지하세요.
INCLUDE_EMPTY_IMAGES = True

# 이미지가 없을 때 기본은 기록만 하고 계속 진행합니다. 엄격하게 중단하려면 True로 바꾸세요.
FAIL_ON_MISSING = False

# COCO bbox가 이미지 밖으로 약간 나간 경우 이미지 경계로 잘라 YOLO label을 만듭니다.
CLIP_BBOX = True

# 기존 output 파일이 있으면 기본은 중단합니다. 같은 OUT_DIR에 다시 쓰려면 True로 바꾸세요.
OVERWRITE = False

# ===== 안전 플래그 =====
DRY_RUN = True   # True이면 복사/저장 없이 통계만 확인합니다.
RUN_PACK = False # 실제 실행하려면 True로 바꾸고, 보통 DRY_RUN=False도 같이 바꿉니다.


## 1. Preview / Dry-run

이미지 개수, annotation 개수, class 개수, 누락 이미지 개수를 먼저 확인합니다. 파일은 만들지 않습니다.


In [ ]:
preview = pack_coco_dataset(
    coco_inputs=COCO_INPUTS,
    out_dir=OUT_DIR,
    label_format=LABEL_FORMAT,
    save_rewritten_coco=SAVE_REWRITTEN_COCO,
    copy_mode=COPY_MODE,
    include_empty_images=INCLUDE_EMPTY_IMAGES,
    fail_on_missing=FAIL_ON_MISSING,
    overwrite=OVERWRITE,
    clip_bbox=CLIP_BBOX,
    dry_run=True,
)
preview.as_dict()


## 2. 실제 packaging 실행

결과 구조는 `OUT_DIR/images/{split}`, `OUT_DIR/labels/{split}`, `OUT_DIR/annotations`, `data.yaml`, `export_manifest.csv`, `export_summary.json`입니다. 실제 실행 전에는 `RUN_PACK=True`, `DRY_RUN=False`로 바꿉니다.


In [ ]:
if RUN_PACK:
    result = pack_coco_dataset(
        coco_inputs=COCO_INPUTS,
        out_dir=OUT_DIR,
        label_format=LABEL_FORMAT,
        save_rewritten_coco=SAVE_REWRITTEN_COCO,
        copy_mode=COPY_MODE,
        include_empty_images=INCLUDE_EMPTY_IMAGES,
        fail_on_missing=FAIL_ON_MISSING,
        overwrite=OVERWRITE,
        clip_bbox=CLIP_BBOX,
        dry_run=DRY_RUN,
    )
    result.as_dict()
else:
    print("RUN_PACK=False 이므로 실제 packaging을 건너뜁니다.")
